# Module 5: Production Batch Evaluation

## Overview

In Module 4, we deployed observability-enabled agents that emit OTEL traces to CloudWatch. Now we need to **evaluate production traffic** at scale using the same evaluation criteria from Module 2.

This notebook implements a **batch evaluation pipeline** that:
1. **Sets up Infrastructure** - Creates Kinesis Firehose and S3 for trace streaming
2. **Exports** - Retrieves OTEL traces from agent runtime log groups
3. **Transforms** - Converts OTEL spans to evaluation test cases
4. **Evaluates** - Runs Strands eval (same evaluators as Module 2)
5. **Stores** - Saves results to S3 and CloudWatch metrics
6. **Drift Detection** - Compares production scores against Module 2 baseline
7. **Feedback Loop** - Converts production failures into new offline test cases

## Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                   Production Batch Evaluation Pipeline                       │
│                                                                              │
│  ┌──────────────────┐                                                       │
│  │ Agent Runtime    │                                                       │
│  │  Log Groups      │                                                       │
│  │ (strands.tracer) │                                                       │
│  └────────┬─────────┘                                                       │
│           │                                                                  │
│           │  Subscription Filter                                            │
│           │  { $.scope.name = "strands.telemetry.tracer" &&                 │
│           │    $.body.output.messages[0].content.finish_reason = "end_turn" }│
│           ▼                                                                  │
│  ┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐    │
│  │ Kinesis Firehose │────▶│   S3 Bucket      │────▶│   Strands Eval   │    │
│  │ (traces-stream)  │     │ (traces/parquet) │     │   (batch job)    │    │
│  └──────────────────┘     └──────────────────┘     └────────┬─────────┘    │
│                                                              │              │
│                              ┌───────────────────────────────┼──────────┐   │
│                              │                               │          │   │
│                              ▼                               ▼          ▼   │
│                       ┌──────────┐              ┌──────────────┐  ┌─────────┐│
│                       │   S3     │              │  CloudWatch  │  │ Athena  ││
│                       │ Results  │              │   Metrics    │  │ Query   ││
│                       └──────────┘              └──────────────┘  └─────────┘│
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Insight: Agent Runtime Log Groups

**OTEL telemetry events are NOT in `aws/spans`**. They are in agent runtime log groups:
- `/aws/bedrock-agentcore/runtimes/{agent-id}-DEFAULT`
- The events have `scope.name = "strands.telemetry.tracer"`
- Final responses have `finish_reason: "end_turn"`

## Evaluation Metrics (Same as Module 2)

1. **Goal Success** - Did the agent successfully address the request?
2. **Helpfulness** - How helpful was the response?
3. **RBAC Compliance** - Did the agent enforce role-based access control correctly?
4. **Tool Parameter Accuracy** - Did the agent select the right tool with correct parameters?
5. **Policy Compliance** - Did the response follow company policies?
6. **Response Quality** - Overall quality of the response
7. **Customer Satisfaction** - Predicted customer satisfaction

## Prerequisites

- Module 4a completed (observability agents deployed)
- Production traffic generated (invocations with traces)

### Time: ~25 minutes

## Step 1: Environment Setup

In [ ]:
import boto3
import json
import os
import time
import pandas as pd
from datetime import datetime, timezone, timedelta
from typing import Dict, Any, List, Optional, Tuple
from botocore.exceptions import ClientError

# Try to load REGION from previous modules
try:
    %store -r REGION
    print(f"✅ Loaded REGION from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using session region: {REGION}")

# Initialize AWS clients
logs_client = boto3.client('logs', region_name=REGION)
s3_client = boto3.client('s3', region_name=REGION)
cloudwatch_client = boto3.client('cloudwatch', region_name=REGION)
sts_client = boto3.client('sts', region_name=REGION)
firehose_client = boto3.client('firehose', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']

# Configuration
TRACES_BUCKET = f'ecommerce-workshop-traces-{ACCOUNT_ID}-{REGION}'
RESULTS_BUCKET = f'ecommerce-workshop-eval-{ACCOUNT_ID}-{REGION}'
FIREHOSE_STREAM_NAME = 'ecommerce-workshop-traces-stream'
CLOUDWATCH_NAMESPACE = 'EcommerceWorkshop/BatchEvaluation'
LOOKBACK_HOURS = 24  # How far back to look for traces

print(f"\n📋 Configuration:")
print(f"   Account ID: {ACCOUNT_ID}")
print(f"   Region: {REGION}")
print(f"   Traces Bucket: {TRACES_BUCKET}")
print(f"   Results Bucket: {RESULTS_BUCKET}")
print(f"   Firehose Stream: {FIREHOSE_STREAM_NAME}")
print(f"   Lookback Hours: {LOOKBACK_HOURS}")

## Step 2: Set Up Streaming Infrastructure

Create the streaming pipeline components:
1. S3 bucket for traces
2. IAM role for Firehose
3. Kinesis Firehose delivery stream
4. CloudWatch Logs subscription filter

In [ ]:
def create_s3_bucket(bucket_name: str) -> bool:
    """Create S3 bucket if it doesn't exist."""
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f"✅ S3 bucket exists: {bucket_name}")
        return True
    except ClientError:
        try:
            if REGION == 'us-east-1':
                s3_client.create_bucket(Bucket=bucket_name)
            else:
                s3_client.create_bucket(
                    Bucket=bucket_name,
                    CreateBucketConfiguration={'LocationConstraint': REGION}
                )
            print(f"✅ Created S3 bucket: {bucket_name}")
            return True
        except Exception as e:
            print(f"❌ Error creating bucket: {e}")
            return False


def create_firehose_role() -> str:
    """Create IAM role for Firehose to write to S3."""
    role_name = 'ecommerce-workshop-firehose-role'
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "firehose.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    s3_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                "s3:AbortMultipartUpload",
                "s3:GetBucketLocation",
                "s3:GetObject",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:PutObject"
            ],
            "Resource": [
                f"arn:aws:s3:::{TRACES_BUCKET}",
                f"arn:aws:s3:::{TRACES_BUCKET}/*"
            ]
        }]
    }
    
    try:
        response = iam_client.get_role(RoleName=role_name)
        role_arn = response['Role']['Arn']
        print(f"✅ IAM role exists: {role_name}")
        return role_arn
    except iam_client.exceptions.NoSuchEntityException:
        try:
            response = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description='Role for Firehose to write traces to S3'
            )
            role_arn = response['Role']['Arn']
            
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName='S3WritePolicy',
                PolicyDocument=json.dumps(s3_policy)
            )
            
            print(f"✅ Created IAM role: {role_name}")
            print("   Waiting 10s for role propagation...")
            time.sleep(10)
            return role_arn
        except Exception as e:
            print(f"❌ Error creating role: {e}")
            return ""


def create_cloudwatch_logs_role() -> str:
    """Create IAM role for CloudWatch Logs to write to Firehose."""
    role_name = 'ecommerce-workshop-cw-logs-role'
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": f"logs.{REGION}.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    firehose_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                "firehose:PutRecord",
                "firehose:PutRecordBatch"
            ],
            "Resource": f"arn:aws:firehose:{REGION}:{ACCOUNT_ID}:deliverystream/{FIREHOSE_STREAM_NAME}"
        }]
    }
    
    try:
        response = iam_client.get_role(RoleName=role_name)
        role_arn = response['Role']['Arn']
        print(f"✅ CloudWatch Logs role exists: {role_name}")
        return role_arn
    except iam_client.exceptions.NoSuchEntityException:
        try:
            response = iam_client.create_role(
                RoleName=role_name,
                AssumeRolePolicyDocument=json.dumps(trust_policy),
                Description='Role for CloudWatch Logs to write to Firehose'
            )
            role_arn = response['Role']['Arn']
            
            iam_client.put_role_policy(
                RoleName=role_name,
                PolicyName='FirehoseWritePolicy',
                PolicyDocument=json.dumps(firehose_policy)
            )
            
            print(f"✅ Created CloudWatch Logs role: {role_name}")
            print("   Waiting 10s for role propagation...")
            time.sleep(10)
            return role_arn
        except Exception as e:
            print(f"❌ Error creating role: {e}")
            return ""


def create_firehose_stream(role_arn: str) -> bool:
    """Create Kinesis Firehose delivery stream."""
    try:
        firehose_client.describe_delivery_stream(
            DeliveryStreamName=FIREHOSE_STREAM_NAME
        )
        print(f"✅ Firehose stream exists: {FIREHOSE_STREAM_NAME}")
        return True
    except firehose_client.exceptions.ResourceNotFoundException:
        try:
            firehose_client.create_delivery_stream(
                DeliveryStreamName=FIREHOSE_STREAM_NAME,
                DeliveryStreamType='DirectPut',
                ExtendedS3DestinationConfiguration={
                    'RoleARN': role_arn,
                    'BucketARN': f'arn:aws:s3:::{TRACES_BUCKET}',
                    'Prefix': 'traces/year=!{timestamp:yyyy}/month=!{timestamp:MM}/day=!{timestamp:dd}/hour=!{timestamp:HH}/',
                    'ErrorOutputPrefix': 'errors/',
                    'BufferingHints': {
                        'SizeInMBs': 5,
                        'IntervalInSeconds': 60
                    },
                    'CompressionFormat': 'GZIP',
                    'CloudWatchLoggingOptions': {
                        'Enabled': False
                    }
                }
            )
            print(f"✅ Created Firehose stream: {FIREHOSE_STREAM_NAME}")
            print("   Waiting 30s for stream to become active...")
            time.sleep(30)
            return True
        except Exception as e:
            print(f"❌ Error creating Firehose: {e}")
            return False


print("="*60)
print("SETTING UP STREAMING INFRASTRUCTURE")
print("="*60)

# Create S3 buckets
create_s3_bucket(TRACES_BUCKET)
create_s3_bucket(RESULTS_BUCKET)

# Create IAM roles
FIREHOSE_ROLE_ARN = create_firehose_role()
CW_LOGS_ROLE_ARN = create_cloudwatch_logs_role()

# Create Firehose stream
if FIREHOSE_ROLE_ARN:
    create_firehose_stream(FIREHOSE_ROLE_ARN)

print("\n✅ Infrastructure setup complete")

## Step 3: Create Subscription Filter for Agent Log Groups

The OTEL telemetry events are in agent runtime log groups. We need to:
1. Find the agent runtime log groups
2. Create subscription filters to stream traces to Firehose

In [ ]:
def find_agent_runtime_log_groups() -> List[str]:
    """Find all AgentCore runtime log groups."""
    log_groups = []
    paginator = logs_client.get_paginator('describe_log_groups')
    
    for page in paginator.paginate(logGroupNamePrefix='/aws/bedrock-agentcore/runtimes/'):
        for lg in page.get('logGroups', []):
            log_groups.append(lg['logGroupName'])
    
    return log_groups


def create_subscription_filter(log_group_name: str, cw_logs_role_arn: str) -> bool:
    """
    Create CloudWatch Logs subscription filter for OTEL events.
    
    Filter pattern matches:
    - scope.name = "strands.telemetry.tracer" (Strands OTEL events)
    - finish_reason = "end_turn" (Final responses only)
    """
    filter_name = 'eval-traces-filter'
    
    # Filter for strands.telemetry.tracer events with end_turn finish_reason
    filter_pattern = '{ $.scope.name = "strands.telemetry.tracer" && $.body.output.messages[0].content.finish_reason = "end_turn" }'
    
    try:
        # Check if filter already exists
        response = logs_client.describe_subscription_filters(
            logGroupName=log_group_name,
            filterNamePrefix=filter_name
        )
        
        if response.get('subscriptionFilters'):
            print(f"   ✅ Subscription filter exists for {log_group_name}")
            return True
    except Exception:
        pass
    
    try:
        firehose_arn = f"arn:aws:firehose:{REGION}:{ACCOUNT_ID}:deliverystream/{FIREHOSE_STREAM_NAME}"
        
        logs_client.put_subscription_filter(
            logGroupName=log_group_name,
            filterName=filter_name,
            filterPattern=filter_pattern,
            destinationArn=firehose_arn,
            roleArn=cw_logs_role_arn
        )
        print(f"   ✅ Created subscription filter for {log_group_name}")
        return True
    except Exception as e:
        print(f"   ⚠️ Error creating filter for {log_group_name}: {str(e)[:80]}")
        return False


print("="*60)
print("SETTING UP LOG SUBSCRIPTION FILTERS")
print("="*60)

# Find agent runtime log groups
AGENT_LOG_GROUPS = find_agent_runtime_log_groups()
print(f"\nFound {len(AGENT_LOG_GROUPS)} agent runtime log groups:")
for lg in AGENT_LOG_GROUPS:
    print(f"   • {lg}")

# Create subscription filters
if AGENT_LOG_GROUPS and CW_LOGS_ROLE_ARN:
    print(f"\nCreating subscription filters...")
    for log_group in AGENT_LOG_GROUPS:
        create_subscription_filter(log_group, CW_LOGS_ROLE_ARN)
else:
    print("\n⚠️ No agent log groups found or IAM role not created")
    print("   Ensure agents from Module 4a are deployed")

## Step 4: OTEL Trace Transformer

This module transforms OTEL events into evaluation test cases.

**Key Insight: Tool Calls Are in a Different Scope**

OTEL events for a single trace span multiple scopes:
- **`strands.telemetry.tracer`**: Final response with `finish_reason: end_turn`
- **`opentelemetry.instrumentation.botocore.bedrock-runtime`**: Actual tool calls with `event.name: gen_ai.assistant.message`

Both share the same `traceId`, so we query ALL events for each trace to get:
1. The final response (input/output)
2. The actual tool calls (e.g., `search_products`, `update_inventory`)

**Tool Classification (Single Product Catalog Agent with RBAC):**
- READ tools (all users): `search_products`, `get_product_details`, `check_inventory`, `get_product_recommendations`, `compare_products`, `get_return_policy`
- WRITE/ADMIN tools (admin only): `create_product`, `update_product`, `delete_product`, `update_inventory`, `update_pricing`

In [ ]:
def parse_otel_event(message: str) -> Optional[Dict[str, Any]]:
    """
    Parse an OTEL event from a CloudWatch log message.
    """
    try:
        event = json.loads(message)
        
        scope_name = event.get('scope', {}).get('name', '')
        if scope_name != 'strands.telemetry.tracer':
            return None
        
        body = event.get('body', {})
        if not body or not isinstance(body, dict):
            return None
        
        if 'input' not in body or 'output' not in body:
            return None
        
        return event
        
    except (json.JSONDecodeError, TypeError):
        return None


def extract_user_message(input_messages: List[Dict]) -> str:
    """
    Extract the user message from OTEL input messages.
    """
    for msg in input_messages:
        role = msg.get('role', '')
        if role == 'user':
            content = msg.get('content', {})
            
            if isinstance(content, str):
                return content
            elif isinstance(content, dict):
                inner_content = content.get('content', '')
                if inner_content:
                    try:
                        parsed = json.loads(inner_content)
                        if isinstance(parsed, list) and len(parsed) > 0:
                            for item in parsed:
                                if isinstance(item, dict) and 'text' in item:
                                    return item['text']
                            return inner_content
                    except (json.JSONDecodeError, TypeError):
                        return inner_content
                
                text = content.get('text', '')
                if text:
                    return text
    
    return ''


def extract_assistant_response(output_messages: List[Dict]) -> str:
    """
    Extract assistant response text from OTEL output messages.
    """
    response_text = ''
    
    for msg in output_messages:
        role = msg.get('role', '')
        if role != 'assistant':
            continue
        
        content = msg.get('content', {})
        if not isinstance(content, dict):
            continue
        
        message_str = content.get('message', '')
        if not message_str:
            continue
        
        try:
            message_array = json.loads(message_str)
            if not isinstance(message_array, list):
                response_text = message_str
                continue
            
            text_parts = []
            for item in message_array:
                if isinstance(item, dict):
                    if 'text' in item:
                        text_parts.append(item['text'])
                elif isinstance(item, str):
                    text_parts.append(item)
            
            if text_parts:
                response_text = '\n'.join(text_parts)
                
        except (json.JSONDecodeError, TypeError):
            response_text = message_str
    
    return response_text


def is_final_response(event: Dict[str, Any]) -> bool:
    """
    Check if this OTEL event represents the final response.
    """
    body = event.get('body', {})
    output = body.get('output', {})
    messages = output.get('messages', [])
    
    for msg in messages:
        content = msg.get('content', {})
        if isinstance(content, dict):
            finish_reason = content.get('finish_reason', '')
            if finish_reason == 'end_turn':
                return True
    
    return False


def extract_tool_calls_from_trace_events(events: List[Dict]) -> List[Dict]:
    """
    Extract actual tool calls from all events in a trace.
    
    Tool calls are found in events with:
    - scope: opentelemetry.instrumentation.botocore.bedrock-runtime
    - event.name: gen_ai.assistant.message
    - body.tool_calls: array of tool call objects
    """
    tool_calls = []
    
    for event in events:
        try:
            data = json.loads(event) if isinstance(event, str) else event
            
            scope = data.get('scope', {}).get('name', '')
            event_name = data.get('attributes', {}).get('event.name', '')
            
            # Tool calls are in bedrock-runtime scope with gen_ai.assistant.message event
            if 'botocore.bedrock-runtime' in scope and event_name == 'gen_ai.assistant.message':
                body = data.get('body', {})
                
                # Extract from tool_calls array
                for tc in body.get('tool_calls', []):
                    func = tc.get('function', {})
                    tool_name = func.get('name', '')
                    tool_args = func.get('arguments', {})
                    
                    if tool_name:
                        tool_calls.append({
                            'name': tool_name,
                            'input': tool_args,
                            'tool_id': tc.get('id', ''),
                            'type': 'tool_call',
                        })
                
                # Also check content array for toolUse
                for content_item in body.get('content', []):
                    if isinstance(content_item, dict) and 'toolUse' in content_item:
                        tool_use = content_item['toolUse']
                        tool_name = tool_use.get('name', '')
                        if tool_name and not any(t['name'] == tool_name for t in tool_calls):
                            tool_calls.append({
                                'name': tool_name,
                                'input': tool_use.get('input', {}),
                                'tool_id': tool_use.get('toolUseId', ''),
                                'type': 'tool_call',
                            })
                            
        except (json.JSONDecodeError, TypeError, AttributeError):
            continue
    
    return tool_calls


# Tool classification for single Product Catalog Agent with RBAC
READ_TOOLS = {"search_products", "get_product_details", "check_inventory",
              "get_product_recommendations", "compare_products", "get_return_policy"}
WRITE_TOOLS = {"create_product", "update_product", "delete_product",
               "update_inventory", "update_pricing"}

def classify_tool_calls(tool_calls):
    """
    Classify tool calls into READ or WRITE categories for the Product Catalog Agent.
    
    Strips any prefix (e.g., ProductTools___) before matching to handle
    both prefixed and unprefixed tool names from OTEL traces.
    
    Returns a set of categories: {'READ'}, {'WRITE'}, {'READ', 'WRITE'}, or empty set.
    """
    categories = set()
    for tool in tool_calls:
        tool_name = tool.split("___")[-1] if "___" in tool else tool
        if tool_name in READ_TOOLS:
            categories.add("READ")
        elif tool_name in WRITE_TOOLS:
            categories.add("WRITE")
    return categories


def get_all_trace_events(log_group: str, trace_id: str, start_time: int, end_time: int) -> List[Dict]:
    """
    Query all events for a specific trace ID to get tool calls.
    """
    try:
        response = logs_client.filter_log_events(
            logGroupName=log_group,
            filterPattern=f'{{ $.traceId = "{trace_id}" }}',
            startTime=start_time,
            endTime=end_time,
            limit=100
        )
        
        events = []
        for event in response.get('events', []):
            try:
                data = json.loads(event.get('message', ''))
                events.append(data)
            except:
                pass
        
        return events
    except Exception as e:
        return []


def transform_otel_to_test_case(
    event: Dict[str, Any], 
    tool_calls: List[Dict] = None
) -> Optional[Dict[str, Any]]:
    """
    Transform an OTEL event into an evaluation test case.
    
    Args:
        event: The strands.telemetry.tracer event with final response
        tool_calls: List of tool calls extracted from the full trace
    """
    try:
        body = event.get('body', {})
        attributes = event.get('attributes', {})
        
        input_data = body.get('input', {})
        input_messages = input_data.get('messages', [])
        
        output_data = body.get('output', {})
        output_messages = output_data.get('messages', [])
        
        user_message = extract_user_message(input_messages)
        if not user_message:
            return None
        
        assistant_response = extract_assistant_response(output_messages)
        if not assistant_response:
            return None
        
        # Use provided tool calls or empty list
        all_tools = tool_calls if tool_calls else []
        
        # Classify tool calls into READ/WRITE categories
        tool_names = [t.get('name', '') for t in all_tools]
        categories = classify_tool_calls(tool_names)
        
        trace_id = event.get('traceId', '')
        span_id = event.get('spanId', '')
        session_id = attributes.get('session.id', '')
        
        time_unix_nano = event.get('timeUnixNano', 0)
        if time_unix_nano:
            timestamp = datetime.fromtimestamp(
                time_unix_nano / 1e9, tz=timezone.utc
            ).isoformat().replace('+00:00', 'Z')
        else:
            timestamp = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
        
        return {
            'trace_id': trace_id,
            'span_id': span_id,
            'session_id': session_id,
            'input': user_message,
            'actual_output': assistant_response,
            'tools_called': all_tools,
            'tool_categories': sorted(categories),
            'timestamp': timestamp,
        }
        
    except Exception as e:
        print(f"Error transforming event: {e}")
        return None


print("OTEL Transformer functions defined:")
print("   - parse_otel_event: Parse strands.telemetry.tracer events")
print("   - extract_user_message: Extract user input from messages")
print("   - extract_assistant_response: Extract final assistant response")
print("   - is_final_response: Check for finish_reason: end_turn")
print("   - extract_tool_calls_from_trace_events: Extract tools from bedrock-runtime events")
print("   - classify_tool_calls: Classify tools as READ or WRITE (Product Catalog Agent RBAC)")
print("   - get_all_trace_events: Query all events for a trace ID")
print("   - transform_otel_to_test_case: Convert to evaluation format")
print("")
print("Tool Classification (Product Catalog Agent):")
print(f"   READ tools:  {sorted(READ_TOOLS)}")
print(f"   WRITE tools: {sorted(WRITE_TOOLS)}")
print("")
print("Key insight: Tool calls are in a DIFFERENT scope than final responses!")
print("   Final response: scope='strands.telemetry.tracer'")
print("   Tool calls: scope='opentelemetry.instrumentation.botocore.bedrock-runtime'")
print("   Both share the same traceId, so we query all events per trace")

## Step 5: Export Traces from Agent Runtime Log Groups

Query the agent runtime log groups directly to retrieve production traces.

In [ ]:
def export_traces_from_log_groups(
    log_groups: List[str],
    lookback_hours: int = 24,
    max_traces: int = 100,
    fetch_tool_calls: bool = True
) -> List[Dict[str, Any]]:
    """
    Export OTEL traces from agent runtime log groups.
    
    This function:
    1. Finds final response events (strands.telemetry.tracer with finish_reason: end_turn)
    2. For each trace, queries ALL events to extract tool calls
    3. Tool calls are in 'opentelemetry.instrumentation.botocore.bedrock-runtime' scope
    
    Args:
        log_groups: List of log group names to query
        lookback_hours: How far back to look for traces
        max_traces: Maximum number of traces to return
        fetch_tool_calls: If True, query all events per trace to get tool calls
        
    Returns:
        List of test case dicts ready for evaluation
    """
    print(f"\nExporting traces from agent runtime log groups...")
    print(f"  Log groups: {len(log_groups)}")
    print(f"  Lookback: {lookback_hours} hours")
    print(f"  Max traces: {max_traces}")
    print(f"  Fetch tool calls: {fetch_tool_calls}")
    
    end_time = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_time = int((datetime.now(timezone.utc) - timedelta(hours=lookback_hours)).timestamp() * 1000)
    
    test_cases = []
    trace_ids_seen = set()
    
    for log_group in log_groups:
        if len(test_cases) >= max_traces:
            break
        
        try:
            # First, filter for final response events
            response = logs_client.filter_log_events(
                logGroupName=log_group,
                filterPattern='{ $.scope.name = "strands.telemetry.tracer" && $.body.output.messages[0].content.finish_reason = "end_turn" }',
                startTime=start_time,
                endTime=end_time,
                limit=50
            )
            
            events = response.get('events', [])
            if not events:
                continue
            
            print(f"\n  {log_group.split('/')[-1]}: {len(events)} final response events")
            
            for event in events:
                if len(test_cases) >= max_traces:
                    break
                
                message = event.get('message', '')
                
                # Parse the OTEL event
                otel_event = parse_otel_event(message)
                if not otel_event:
                    continue
                
                # Avoid duplicates
                trace_id = otel_event.get('traceId', '')
                if trace_id in trace_ids_seen:
                    continue
                trace_ids_seen.add(trace_id)
                
                # Fetch all events for this trace to get tool calls
                tool_calls = []
                if fetch_tool_calls and trace_id:
                    all_trace_events = get_all_trace_events(
                        log_group, trace_id, start_time, end_time
                    )
                    tool_calls = extract_tool_calls_from_trace_events(all_trace_events)
                
                # Transform to test case with tool calls
                test_case = transform_otel_to_test_case(otel_event, tool_calls)
                if test_case:
                    test_cases.append(test_case)
                    
                    # Show progress for traces with tools
                    if tool_calls:
                        tools_str = ', '.join([t['name'] for t in tool_calls[:3]])
                        if len(tool_calls) > 3:
                            tools_str += f" (+{len(tool_calls)-3} more)"
                        categories = test_case.get('tool_categories', [])
                        cat_str = '/'.join(categories) if categories else 'NONE'
                        print(f"    Trace {trace_id[:12]}: {len(tool_calls)} tools [{tools_str}] ({cat_str})")
                
        except Exception as e:
            print(f"  Error querying {log_group.split('/')[-1]}: {str(e)[:50]}")
    
    return test_cases


# Export traces from agent log groups
print("="*60)
print("EXPORTING PRODUCTION TRACES")
print("="*60)

if AGENT_LOG_GROUPS:
    PRODUCTION_TRACES = export_traces_from_log_groups(
        log_groups=AGENT_LOG_GROUPS,
        lookback_hours=LOOKBACK_HOURS,
        max_traces=20,
        fetch_tool_calls=True  # Query all events to get actual tool calls
    )
else:
    PRODUCTION_TRACES = []
    print("\nNo agent log groups found")

print(f"\n{'='*60}")
print(f"Export Summary")
print(f"{'='*60}")
print(f"   Total test cases: {len(PRODUCTION_TRACES)}")

# Count traces with tool calls
traces_with_tools = sum(1 for t in PRODUCTION_TRACES if t.get('tools_called'))
print(f"   Traces with tool calls: {traces_with_tools}")

# Show tool category distribution
if PRODUCTION_TRACES:
    cat_counts = {}
    for trace in PRODUCTION_TRACES:
        cats = trace.get('tool_categories', [])
        key = '/'.join(cats) if cats else 'NONE'
        cat_counts[key] = cat_counts.get(key, 0) + 1
    
    print(f"\n   Tool category distribution:")
    for cat, count in sorted(cat_counts.items()):
        pct = count / len(PRODUCTION_TRACES) * 100
        print(f"      {cat}: {count} ({pct:.0f}%)")

if PRODUCTION_TRACES:
    print(f"\n   Sample test case:")
    sample = PRODUCTION_TRACES[0]
    print(f"   Trace ID: {sample['trace_id'][:20]}...")
    print(f"   Input: {sample['input'][:60]}..." if len(sample['input']) > 60 else f"   Input: {sample['input']}")
    print(f"   Output: {sample['actual_output'][:60]}..." if len(sample['actual_output']) > 60 else f"   Output: {sample['actual_output']}")
    tools = sample.get('tools_called', [])
    if tools:
        print(f"   Tools ({len(tools)}): {[t['name'] for t in tools[:3]]}")
    print(f"   Categories: {sample.get('tool_categories', [])}")

In [ ]:
# Validate exported traces before running evaluation
print("=" * 60)
print("TRACE QUALITY VALIDATION")
print("=" * 60)

if PRODUCTION_TRACES:
    traces_with_tools = sum(1 for t in PRODUCTION_TRACES if t.get('tools_called'))
    traces_with_output = sum(1 for t in PRODUCTION_TRACES if t.get('actual_output', '').strip())
    
    print(f"  Total traces: {len(PRODUCTION_TRACES)}")
    print(f"  With tool calls: {traces_with_tools}")
    print(f"  With output: {traces_with_output}")
    
    if traces_with_tools == 0:
        print(f"\n⚠️  WARNING: No traces have tool calls!")
        print(f"  Evaluators like tool_parameter_accuracy will score 0.")
        print(f"  This may explain high failure rates in batch evaluation.")
    
    if traces_with_output < len(PRODUCTION_TRACES) * 0.5:
        print(f"\n⚠️  WARNING: Less than 50% of traces have output.")
    
    if traces_with_tools > 0 and traces_with_output > len(PRODUCTION_TRACES) * 0.5:
        print(f"\n✅ Traces look good for evaluation.")
else:
    print("  No traces exported. Check the log groups and time range.")


## Step 6: Alternative - Export from S3 (Firehose Data)

If Firehose has been running, we can also query from S3 for historical data.

In [ ]:
import gzip

def export_traces_from_s3(
    bucket: str,
    prefix: str = 'traces/',
    lookback_hours: int = 24,
    max_traces: int = 100
) -> List[Dict[str, Any]]:
    """
    Export traces from S3 (Firehose destination).
    
    Args:
        bucket: S3 bucket name
        prefix: S3 prefix for trace files
        lookback_hours: How far back to look
        max_traces: Maximum traces to return
        
    Returns:
        List of test case dicts
    """
    print(f"\nExporting traces from S3...")
    print(f"  Bucket: {bucket}")
    print(f"  Prefix: {prefix}")
    
    test_cases = []
    trace_ids_seen = set()
    
    # Generate prefixes for the lookback period
    now = datetime.now(timezone.utc)
    prefixes_to_check = []
    
    for hours_ago in range(lookback_hours + 1):
        dt = now - timedelta(hours=hours_ago)
        date_prefix = f"{prefix}year={dt.year}/month={dt.month:02d}/day={dt.day:02d}/hour={dt.hour:02d}/"
        prefixes_to_check.append(date_prefix)
    
    try:
        for s3_prefix in prefixes_to_check:
            if len(test_cases) >= max_traces:
                break
            
            try:
                response = s3_client.list_objects_v2(
                    Bucket=bucket,
                    Prefix=s3_prefix,
                    MaxKeys=20
                )
                
                for obj in response.get('Contents', []):
                    if len(test_cases) >= max_traces:
                        break
                    
                    key = obj['Key']
                    
                    # Download and parse file
                    try:
                        obj_response = s3_client.get_object(Bucket=bucket, Key=key)
                        body = obj_response['Body'].read()
                        
                        # Handle gzip compression
                        if key.endswith('.gz'):
                            body = gzip.decompress(body)
                        
                        content = body.decode('utf-8')
                        
                        # Parse each line as JSON
                        for line in content.split('\n'):
                            if not line.strip():
                                continue
                            
                            if len(test_cases) >= max_traces:
                                break
                            
                            otel_event = parse_otel_event(line)
                            if not otel_event:
                                continue
                            
                            if not is_final_response(otel_event):
                                continue
                            
                            trace_id = otel_event.get('traceId', '')
                            if trace_id in trace_ids_seen:
                                continue
                            trace_ids_seen.add(trace_id)
                            
                            test_case = transform_otel_to_test_case(otel_event)
                            if test_case:
                                test_cases.append(test_case)
                                
                    except Exception as e:
                        print(f"  Error reading {key}: {str(e)[:50]}")
                        
            except Exception:
                pass  # Prefix may not exist yet
        
        if test_cases:
            print(f"  ✅ Extracted {len(test_cases)} test cases from S3")
        else:
            print(f"  No traces found in S3 yet (Firehose may need time to buffer)")
            
    except ClientError as e:
        print(f"  S3 bucket not accessible: {str(e)[:50]}")
    
    return test_cases


# Try to get additional traces from S3 if we need more
if len(PRODUCTION_TRACES) < 10:
    print("\n--- Checking S3 for additional traces ---")
    s3_traces = export_traces_from_s3(
        bucket=TRACES_BUCKET,
        prefix='traces/',
        lookback_hours=LOOKBACK_HOURS,
        max_traces=20 - len(PRODUCTION_TRACES)
    )
    
    # Merge unique traces
    existing_ids = {t['trace_id'] for t in PRODUCTION_TRACES}
    for trace in s3_traces:
        if trace['trace_id'] not in existing_ids:
            PRODUCTION_TRACES.append(trace)
    
    print(f"\n📊 Updated total: {len(PRODUCTION_TRACES)} test cases")

## Step 7: Import Custom Evaluators

Import the same evaluators from Module 2 for consistency. These are defined in `custom_evaluators.py`.

The evaluators align with the single Product Catalog Agent architecture:
- **GoalSuccessEvaluator** - Did the agent address the request?
- **HelpfulnessEvaluator** - How helpful was the response?
- **RBACComplianceEvaluator** - Did the agent enforce role-based access control?
- **ToolParameterAccuracyEvaluator** - Was the right tool used with correct parameters?
- **PolicyComplianceEvaluator** - Did the response follow company policies?
- **ResponseQualityEvaluator** - Overall response quality
- **CustomerSatisfactionEvaluator** - Predicted customer satisfaction

In [ ]:
import sys
sys.path.insert(0, '.')

from strands_evals import Experiment, Case

# Import custom evaluators aligned with single Product Catalog Agent + RBAC
from custom_evaluators import (
    GoalSuccessEvaluator,
    HelpfulnessEvaluator,
    RBACComplianceEvaluator,
    ToolParameterAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator,
    get_all_custom_evaluators,
)

# Get all evaluators as a dictionary
EVALUATORS = get_all_custom_evaluators()

print(f"Loaded {len(EVALUATORS)} evaluators from custom_evaluators.py:")
for name in EVALUATORS:
    print(f"   - {name}")

## Step 8: Convert to Strands Eval Cases and Run Evaluation

In [ ]:
def convert_traces_to_cases(traces: List[Dict[str, Any]]) -> List[Case]:
    """
    Convert production traces to strands-evals Case objects.
    
    Key insight: for production evaluation, we already HAVE the output.
    We don't need to re-invoke the agent - we evaluate the cached output.
    """
    cases = []
    
    for trace in traces:
        case = Case(
            name=trace['trace_id'][:20],
            input=trace['input'],
            expected_output=trace['actual_output'],
            metadata={
                'trace_id': trace['trace_id'],
                'session_id': trace.get('session_id', ''),
                'timestamp': trace['timestamp'],
                'tools_called': trace.get('tools_called', []),
                'tool_categories': trace.get('tool_categories', []),
            }
        )
        cases.append(case)
    
    return cases


# Convert traces to cases
if PRODUCTION_TRACES:
    EVAL_CASES = convert_traces_to_cases(PRODUCTION_TRACES)
    print(f"Converted {len(EVAL_CASES)} traces to evaluation cases")
    
    # Create response cache
    RESPONSE_CACHE = {}
    for trace in PRODUCTION_TRACES:
        case_name = trace['trace_id'][:20]
        RESPONSE_CACHE[case_name] = trace['actual_output']
    
    print(f"Cached {len(RESPONSE_CACHE)} production responses")
    
    # Show sample test cases with tool categories
    print(f"\n{'='*70}")
    print("SAMPLE TEST CASES WITH TOOL CLASSIFICATION")
    print(f"{'='*70}")
    
    num_samples = min(3, len(EVAL_CASES))
    for i in range(num_samples):
        case = EVAL_CASES[i]
        trace = PRODUCTION_TRACES[i]
        categories = trace.get('tool_categories', [])
        
        print(f"\n{'_'*70}")
        print(f"Test Case {i+1} of {len(EVAL_CASES)}")
        print(f"{'_'*70}")
        print(f"Trace ID:  {case.name}")
        print(f"Timestamp: {case.metadata.get('timestamp', 'N/A')}")
        
        # Show input
        input_display = case.input if len(case.input) <= 80 else case.input[:80] + "..."
        print(f"\nINPUT: {input_display}")
        
        # Show tool classification
        print(f"\nTOOL CLASSIFICATION:")
        print(f"   Categories: {', '.join(categories) if categories else 'NONE'}")
        
        tools_called = trace.get('tools_called', [])
        if tools_called:
            print(f"   Tools Used ({len(tools_called)}):")
            for tool in tools_called[:5]:
                tool_name = tool.get('name', '')
                # Show stripped name and category
                stripped = tool_name.split("___")[-1] if "___" in tool_name else tool_name
                cat = "READ" if stripped in READ_TOOLS else ("WRITE" if stripped in WRITE_TOOLS else "UNKNOWN")
                print(f"      - {tool_name} [{cat}]")
            if len(tools_called) > 5:
                print(f"      ... and {len(tools_called) - 5} more")
        
        # Show output (truncate)
        output_display = case.expected_output if len(case.expected_output) <= 120 else case.expected_output[:120] + "..."
        print(f"\nOUTPUT: {output_display}")
    
    print(f"\n{'='*70}")
    print(f"Showing {num_samples} of {len(EVAL_CASES)} test cases")
    print(f"{'='*70}")
    
else:
    EVAL_CASES = []
    RESPONSE_CACHE = {}
    print("\nNo production traces to convert")
    print("   Please ensure:")
    print("   1. Agents from Module 4a are deployed")
    print("   2. Agents have been invoked (generate some test traffic)")
    print("   3. Wait a few minutes for logs to be available")



# ── Trace quality validation ──
if EVAL_CASES:
    no_tool_cases = sum(
        1 for c in EVAL_CASES 
        if isinstance(c.metadata, dict) and not c.metadata.get('tools_called')
    )
    total = len(EVAL_CASES)
    if no_tool_cases > total * 0.5:
        print(f"\n⚠️  WARNING: {no_tool_cases}/{total} traces have NO tool calls.")
        print("   Production agent may have had Gateway/MCP connectivity issues.")
        print("   Evaluation scores will be low — this is expected behavior, not evaluator bugs.")
    elif no_tool_cases > 0:
        print(f"\n📊 Note: {no_tool_cases}/{total} traces have no tool calls.")


def cached_task(case) -> str:
    """Task function that returns cached production response."""
    return RESPONSE_CACHE.get(case.name, "Error: Response not found in cache")

> ⏱ **Timing note:** The next cell runs 7 evaluators across all production traces,
> making ~140 LLM calls to the judge model. It typically takes **8-15 minutes**.
> This is normal — you'll see progress updates as each evaluator completes.


In [ ]:
# Run all evaluators on production traces
print("="*60)
print("RUNNING BATCH EVALUATION")
print("="*60)

EVALUATION_RESULTS = {}

if EVAL_CASES:
    for evaluator_name, evaluator in EVALUATORS.items():
        print(f"\n--- Running {evaluator_name} evaluation ---")
        
        experiment = Experiment(
            cases=EVAL_CASES,
            evaluators=[evaluator]
        )
        
        # Use async method — Jupyter already runs an event loop so asyncio.run() fails
        results = await experiment.run_evaluations_async(cached_task)
        report = results[0]
        
        EVALUATION_RESULTS[evaluator_name] = {
            'scores': report.scores,
            'reasons': report.reasons,
            'overall_score': report.overall_score,
            'pass_rate': sum(report.test_passes) / len(report.test_passes) if report.test_passes else 0,
        }
        
        print(f"   Overall Score: {report.overall_score:.2f}")
        print(f"   Pass Rate: {sum(report.test_passes)}/{len(report.test_passes)}")
        
        time.sleep(1)  # Rate limiting
else:
    print("\nNo evaluation cases to process")
    print("   Generate some agent traffic first, then re-run this notebook")

## Step 9: Aggregate and Display Results

In [ ]:
print("\n" + "="*60)
print("BATCH EVALUATION SUMMARY")
print("="*60)

if EVALUATION_RESULTS:
    print(f"\nTotal traces evaluated: {len(EVAL_CASES)}")
    print(f"Time range: Last {LOOKBACK_HOURS} hours")
    print(f"\n{'Metric':<25} {'Overall Score':<15} {'Pass Rate':<15}")
    print("-"*55)
    
    for metric_name, result in EVALUATION_RESULTS.items():
        overall = result['overall_score']
        pass_rate = result['pass_rate']
        print(f"{metric_name:<25} {overall:>10.1%}      {pass_rate:>10.1%}")
    
    # Calculate aggregate metrics
    aggregate_metrics = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'traces_evaluated': len(EVAL_CASES),
        'lookback_hours': LOOKBACK_HOURS,
    }
    
    for metric_name, result in EVALUATION_RESULTS.items():
        aggregate_metrics[f'{metric_name}_score'] = result['overall_score']
        aggregate_metrics[f'{metric_name}_pass_rate'] = result['pass_rate']
    
    print("\n" + "="*60)
else:
    print("\n⚠️ No evaluation results to display")
    aggregate_metrics = {}

#### Interpreting Production Evaluation Scores

Compare these production scores against the Module 02 baseline to understand agent behavior in the wild:

- **`helpfulness` and `response_quality` at ~100%**: The agent consistently produces well-structured, informative responses. These "ceiling" metrics are useful for regression detection — any drop signals a problem.
- **`tool_parameter_accuracy` (~70%)**: Lower than other metrics because some production traces show the agent answering from general knowledge instead of calling tools. This is a real finding — the agent should be using `search_products` for search queries, not generating answers without evidence.
- **`rbac_compliance` (~92%)**: Slightly below 1.0 because production traffic includes edge cases not covered in the Module 02 test suite — exactly why production evaluation matters.
- **`customer_satisfaction` (~87%)**: Lower for traces where the agent correctly refused unauthorized operations. As noted in Module 02b: security and satisfaction can be inversely correlated.

**Key principle:** Production scores are often slightly lower than baseline scores, because production traffic is messier and more diverse than curated test cases. A 5–10% drop is normal; a >10% drop triggers a drift alert.

In [ ]:
# Create detailed results DataFrame with tool categories
if EVALUATION_RESULTS and EVAL_CASES:
    rows = []
    
    for i, case in enumerate(EVAL_CASES):
        trace = PRODUCTION_TRACES[i]
        categories = trace.get('tool_categories', [])
        tools_called = trace.get('tools_called', [])
        
        row = {
            'trace_id': case.name,
            'timestamp': case.metadata.get('timestamp', ''),
            'input': trace['input'][:80] + '...' if len(trace['input']) > 80 else trace['input'],
            'output': trace['actual_output'][:100] + '...' if len(trace['actual_output']) > 100 else trace['actual_output'],
            'tool_categories': '/'.join(categories) if categories else 'NONE',
            'tools_used': ', '.join([t.get('name', '') for t in tools_called[:3]]) or 'None',
            'num_tools': len(tools_called),
        }
        
        for metric_name, result in EVALUATION_RESULTS.items():
            row[metric_name] = result['scores'][i] if i < len(result['scores']) else None
        
        rows.append(row)
    
    results_df = pd.DataFrame(rows)
    
    print("Detailed Results with Tool Classification (first 10 rows):")
    print("-" * 120)
    
    # Display with better formatting
    pd.set_option('display.max_colwidth', 40)
    display_cols = ['trace_id', 'input', 'output', 'tool_categories', 'tools_used'] + list(EVALUATORS.keys())
    print(results_df[display_cols].head(10).to_string(index=False))
    
    # Tool category summary
    print(f"\nTool Category Distribution:")
    cat_counts = results_df['tool_categories'].value_counts()
    for cat, count in cat_counts.items():
        pct = count / len(results_df) * 100
        print(f"   {cat}: {count} ({pct:.1f}%)")
    
    print(f"\nScore Distribution:")
    for metric_name in EVALUATORS.keys():
        if metric_name in results_df.columns:
            scores = results_df[metric_name].dropna()
            if len(scores) > 0:
                print(f"   {metric_name}: mean={scores.mean():.2f}, min={scores.min():.2f}, max={scores.max():.2f}")
else:
    results_df = pd.DataFrame()

## Step 10: Store Results

In [ ]:
# Save results locally
RESULTS_DIR = 'evaluation_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')

if EVALUATION_RESULTS:
    # Build comprehensive results with all data together
    full_results = []
    for i, case in enumerate(EVAL_CASES):
        trace = PRODUCTION_TRACES[i]
        categories = trace.get('tool_categories', [])
        tools_called = trace.get('tools_called', [])
        
        result_entry = {
            # Trace metadata
            'trace_id': trace['trace_id'],
            'session_id': trace.get('session_id', ''),
            'timestamp': trace['timestamp'],
            
            # Input/Output data used in evaluation
            'input': trace['input'],
            'actual_output': trace['actual_output'],
            
            # Tool classification (Product Catalog Agent RBAC)
            'tool_categories': categories,
            'tools_called': tools_called,
            'num_tools': len(tools_called),
            
            # Evaluation results for each metric
            'evaluations': {}
        }
        
        for metric_name, result in EVALUATION_RESULTS.items():
            if i < len(result['scores']):
                result_entry['evaluations'][metric_name] = {
                    'score': result['scores'][i],
                    'reason': result['reasons'][i] if i < len(result['reasons']) else ''
                }
        
        full_results.append(result_entry)
    
    # Save comprehensive JSON file
    full_file = f"{RESULTS_DIR}/batch_eval_full_{timestamp_str}.json"
    with open(full_file, 'w') as f:
        json.dump(full_results, f, indent=2, default=str)
    print(f"Saved comprehensive results to {full_file}")
    
    # Save aggregate metrics
    aggregate_file = f"{RESULTS_DIR}/batch_eval_aggregate_{timestamp_str}.json"
    with open(aggregate_file, 'w') as f:
        json.dump(aggregate_metrics, f, indent=2, default=str)
    print(f"Saved aggregate metrics to {aggregate_file}")
    
    # Save detailed CSV for easy viewing
    if not results_df.empty:
        detailed_file = f"{RESULTS_DIR}/batch_eval_detailed_{timestamp_str}.csv"
        results_df.to_csv(detailed_file, index=False)
        print(f"Saved detailed CSV to {detailed_file}")
    
    # Display sample of comprehensive results
    print(f"\n{'='*70}")
    print("COMPREHENSIVE EVALUATION RESULTS (Sample)")
    print(f"{'='*70}")
    
    num_samples = min(2, len(full_results))
    for i in range(num_samples):
        entry = full_results[i]
        categories = entry.get('tool_categories', [])
        
        print(f"\n{'_'*70}")
        print(f"TRACE {i+1}: {entry['trace_id'][:30]}...")
        print(f"{'_'*70}")
        
        # Input
        input_display = entry['input'] if len(entry['input']) <= 70 else entry['input'][:70] + "..."
        print(f"\nINPUT: {input_display}")
        
        # Tool info
        print(f"\nTOOL INFO:")
        print(f"   Categories: {', '.join(categories) if categories else 'NONE'}")
        tools = entry.get('tools_called', [])
        if tools:
            tools_display = ', '.join([t.get('name', '') for t in tools[:3]])
            if len(tools) > 3:
                tools_display += f" (+{len(tools)-3} more)"
            print(f"   Tools: {tools_display}")
        
        # Output
        output_display = entry['actual_output'] if len(entry['actual_output']) <= 100 else entry['actual_output'][:100] + "..."
        print(f"\nOUTPUT: {output_display}")
        
        # Evaluation scores and reasons
        print(f"\nEVALUATION RESULTS:")
        for metric_name, eval_data in entry['evaluations'].items():
            score = eval_data['score']
            reason = eval_data['reason']
            
            # Score indicator
            if score >= 0.8:
                indicator = "PASS"
            elif score >= 0.5:
                indicator = "WARN"
            else:
                indicator = "FAIL"
            
            print(f"\n   [{indicator}] {metric_name}: {score:.0%}")
            
            # Show reason (truncate if too long)
            if reason:
                reason_display = reason if len(reason) <= 80 else reason[:80] + "..."
                print(f"      Reason: {reason_display}")
    
    print(f"\n{'='*70}")
    print(f"Full results saved to: {full_file}")
    print(f"{'='*70}")
    
else:
    full_results = []
    print("No results to save")

In [ ]:
# Publish metrics to CloudWatch
def publish_metrics_to_cloudwatch(metrics: Dict[str, Any]) -> bool:
    """Publish evaluation metrics to CloudWatch."""
    if not metrics:
        return False
    
    metric_data = []
    timestamp = datetime.now(timezone.utc)
    
    metric_data.append({
        'MetricName': 'TracesEvaluated',
        'Value': metrics.get('traces_evaluated', 0),
        'Unit': 'None',
        'Timestamp': timestamp,
    })
    
    for key, value in metrics.items():
        if key.endswith('_score') and isinstance(value, (int, float)):
            metric_name = key.replace('_score', '_Score')
            metric_data.append({
                'MetricName': metric_name,
                'Value': value * 100,
                'Unit': 'None',
                'Timestamp': timestamp,
            })
        elif key.endswith('_pass_rate') and isinstance(value, (int, float)):
            metric_name = key.replace('_pass_rate', '_PassRate')
            metric_data.append({
                'MetricName': metric_name,
                'Value': value * 100,
                'Unit': 'None',
                'Timestamp': timestamp,
            })
    
    if metric_data:
        try:
            for i in range(0, len(metric_data), 20):
                batch = metric_data[i:i+20]
                cloudwatch_client.put_metric_data(
                    Namespace=CLOUDWATCH_NAMESPACE,
                    MetricData=batch
                )
            print(f"✅ Published {len(metric_data)} metrics to CloudWatch namespace: {CLOUDWATCH_NAMESPACE}")
            return True
        except Exception as e:
            print(f"❌ Error publishing to CloudWatch: {e}")
            return False
    
    return False


if aggregate_metrics:
    publish_metrics_to_cloudwatch(aggregate_metrics)

In [ ]:
# Save to S3
def save_results_to_s3(results: List[Dict], bucket: str, prefix: str = 'batch-evaluations') -> str:
    """Save evaluation results to S3."""
    timestamp_str = datetime.now().strftime('%Y/%m/%d/%H')
    key = f"{prefix}/{timestamp_str}/batch_eval_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
    
    try:
        s3_client.put_object(
            Bucket=bucket,
            Key=key,
            Body=json.dumps(results, indent=2, default=str).encode('utf-8'),
            ContentType='application/json'
        )
        
        print(f"✅ Saved results to s3://{bucket}/{key}")
        return f"s3://{bucket}/{key}"
        
    except Exception as e:
        print(f"❌ Error saving to S3: {e}")
        return ""


if EVALUATION_RESULTS and full_results:
    s3_path = save_results_to_s3(full_results, RESULTS_BUCKET)
else:
    s3_path = ""

## Drift Detection

Compare production evaluation scores against the **Module 02 baseline** to detect quality regressions.

The baseline metrics were saved during offline evaluation in Module 02. By comparing production scores against these baselines, we can detect when the agent's behavior has drifted -- for example, due to model updates, prompt changes, or shifts in user request patterns.

**Drift threshold**: An absolute difference of more than **10%** (0.1) between baseline and production scores triggers an alert.

In [ ]:
# Load baseline metrics from Module 02
BASELINE_PATH = os.path.join('..', '02-evaluation-baseline', 'baseline_metrics.json')

baseline_metrics = {}
try:
    with open(BASELINE_PATH, 'r') as f:
        raw_baseline = json.load(f)

    # baseline_metrics.json has a nested structure:
    #   { "timestamp": ..., "agent": ..., "scores": { metric: float }, "thresholds": {...}, ... }
    # Extract just the scores dict for downstream comparison.
    if 'scores' in raw_baseline and isinstance(raw_baseline['scores'], dict):
        baseline_metrics = raw_baseline['scores']
    else:
        # Fallback: treat as flat dict, keeping only numeric values
        baseline_metrics = {k: v for k, v in raw_baseline.items() if isinstance(v, (int, float))}

    print(f"Loaded baseline metrics from {BASELINE_PATH}")
    print(f"\nBaseline scores ({len(baseline_metrics)} evaluators):")
    for evaluator, score in baseline_metrics.items():
        print(f"   {evaluator}: {score:.2f}")
except FileNotFoundError:
    print(f"WARNING: Baseline metrics not found at {BASELINE_PATH}")
    print("Run Module 02 first to generate baseline_metrics.json")
    print("Using empty baseline (all comparisons will show as drift)")
except json.JSONDecodeError as e:
    print(f"ERROR: Could not parse baseline metrics: {e}")

In [ ]:
# Compute production aggregate scores from batch evaluation results
production_metrics = {}

if EVALUATION_RESULTS:
    for evaluator_name, result in EVALUATION_RESULTS.items():
        production_metrics[evaluator_name] = result['overall_score']
    
    print("Production aggregate scores:")
    for evaluator, score in production_metrics.items():
        print(f"   {evaluator}: {score:.2f}")
else:
    print("No evaluation results available to compute production metrics.")

In [ ]:
# Compare baseline vs production scores with drift threshold
DRIFT_THRESHOLD = 0.1  # 10% drift triggers alert

drift_report = {}

if baseline_metrics and production_metrics:
    for evaluator, baseline_score in baseline_metrics.items():
        prod_score = production_metrics.get(evaluator, 0)
        drift = prod_score - baseline_score
        drift_report[evaluator] = {
            "baseline": baseline_score,
            "production": prod_score,
            "drift": drift,
            "alert": abs(drift) > DRIFT_THRESHOLD
        }
    
    # Also check for evaluators in production that are not in baseline
    for evaluator in production_metrics:
        if evaluator not in drift_report:
            drift_report[evaluator] = {
                "baseline": 0,
                "production": production_metrics[evaluator],
                "drift": production_metrics[evaluator],
                "alert": True  # New evaluator, no baseline
            }
    
    # Count alerts
    alerts = sum(1 for v in drift_report.values() if v['alert'])
    print(f"Drift analysis complete: {len(drift_report)} evaluators compared")
    print(f"Drift threshold: {DRIFT_THRESHOLD:.0%}")
    print(f"Alerts triggered: {alerts}")
elif production_metrics:
    print("No baseline metrics available - skipping drift comparison.")
    print("Run Module 02 to generate baseline_metrics.json, then re-run this section.")
else:
    print("No production metrics available - run batch evaluation first.")

In [ ]:
# Publish drift metrics to CloudWatch
DRIFT_NAMESPACE = 'Workshop/BatchEvaluation'

if drift_report:
    drift_metric_data = []
    timestamp = datetime.now(timezone.utc)
    
    for evaluator, data in drift_report.items():
        # Publish drift value
        drift_metric_data.append({
            'MetricName': f'{evaluator}_Drift',
            'Value': data['drift'] * 100,
            'Unit': 'None',
            'Timestamp': timestamp,
            'Dimensions': [
                {'Name': 'EvaluatorName', 'Value': evaluator},
                {'Name': 'Pipeline', 'Value': 'BatchEvaluation'},
            ]
        })
        
        # Publish alert flag (1 = alert, 0 = ok)
        drift_metric_data.append({
            'MetricName': f'{evaluator}_DriftAlert',
            'Value': 1.0 if data['alert'] else 0.0,
            'Unit': 'None',
            'Timestamp': timestamp,
            'Dimensions': [
                {'Name': 'EvaluatorName', 'Value': evaluator},
                {'Name': 'Pipeline', 'Value': 'BatchEvaluation'},
            ]
        })
    
    try:
        # CloudWatch allows max 20 metrics per PutMetricData call
        for i in range(0, len(drift_metric_data), 20):
            batch = drift_metric_data[i:i+20]
            cloudwatch_client.put_metric_data(
                Namespace=DRIFT_NAMESPACE,
                MetricData=batch
            )
        print(f"Published {len(drift_metric_data)} drift metrics to CloudWatch namespace: {DRIFT_NAMESPACE}")
    except Exception as e:
        print(f"Error publishing drift metrics to CloudWatch: {e}")
else:
    print("No drift report to publish.")

In [ ]:
# Display drift report as a formatted table
if drift_report:
    print("="*80)
    print("DRIFT DETECTION REPORT: Baseline (Module 02) vs Production")
    print("="*80)
    print(f"\n{'Evaluator':<30} {'Baseline':>10} {'Production':>12} {'Drift':>10} {'Status':>10}")
    print("-"*80)
    
    for evaluator, data in drift_report.items():
        baseline_str = f"{data['baseline']:.1%}"
        prod_str = f"{data['production']:.1%}"
        drift_str = f"{data['drift']:+.1%}"
        status = "ALERT" if data['alert'] else "OK"
        
        print(f"{evaluator:<30} {baseline_str:>10} {prod_str:>12} {drift_str:>10} {status:>10}")
    
    alerts = [k for k, v in drift_report.items() if v['alert']]
    print(f"\n{'='*80}")
    if alerts:
        print(f"\nALERTS ({len(alerts)}):")
        for evaluator in alerts:
            data = drift_report[evaluator]
            direction = "improved" if data['drift'] > 0 else "regressed"
            print(f"   - {evaluator}: {direction} by {abs(data['drift']):.1%} (threshold: {DRIFT_THRESHOLD:.0%})")
        print(f"\nAction: Investigate root cause of drift. Check for model updates, prompt changes,")
        print(f"or shifts in user request patterns.")
    else:
        print("\nNo drift alerts. All evaluators are within the {:.0%} threshold.".format(DRIFT_THRESHOLD))
else:
    print("No drift report available. Ensure both baseline and production metrics exist.")

In [ ]:
# Visualize drift: Baseline vs Production scores
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend for notebooks
    %matplotlib inline
    
    if drift_report:
        evaluators = list(drift_report.keys())
        baseline_scores = [drift_report[e]['baseline'] for e in evaluators]
        prod_scores = [drift_report[e]['production'] for e in evaluators]
        
        # Shorten names for display
        short_names = [e.replace('_', '\n') for e in evaluators]
        
        fig, ax = plt.subplots(figsize=(12, 5))
        x = range(len(evaluators))
        width = 0.35
        
        bars1 = ax.bar([i - width/2 for i in x], baseline_scores, width,
                       label='Module 02 Baseline', color='#2196F3', alpha=0.8)
        bars2 = ax.bar([i + width/2 for i in x], prod_scores, width,
                       label='Production', color='#FF9800', alpha=0.8)
        
        ax.set_ylabel('Score')
        ax.set_title('Drift Detection: Baseline vs Production Evaluation Scores')
        ax.set_xticks(list(x))
        ax.set_xticklabels(short_names, fontsize=8)
        ax.legend()
        ax.set_ylim(0, 1.1)
        ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.3, label='90% threshold')
        
        # Add score labels on bars
        for bar in bars1:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)
        
        plt.tight_layout()
        plt.show()
        print('\n📊 Chart shows baseline (blue) vs production (orange) scores.')
        print('   Bars below the green dashed line (90%) may need attention.')
    else:
        print('No drift data available for visualization.')
        
except ImportError:
    print('matplotlib not available — drift table above shows the same data.')


#### What Drift Tells You

Drift is the difference between **expected performance** (Module 02 baseline) and **actual performance** (production batch evaluation). But not all drift is bad:

- **Positive drift** (production > baseline): The agent performs *better* in production than in testing. This happens when production queries are simpler than adversarial test cases, or when the model has improved.
- **Negative drift** (production < baseline): The agent performs *worse* in production. Investigate: are users asking harder questions? Did the model change? Is the data stale?
- **Alerts (>10% change)**: These need investigation regardless of direction. Even positive drift can signal an issue — e.g., if `rbac_compliance` jumps from 0.9 to 1.0, it might mean the evaluator is no longer catching violations (false negatives).

**The feedback loop:** Drift alerts trigger investigation. Investigation finds failed traces. Failed traces become new test cases (see the Production Feedback Loop section below). New test cases strengthen the Module 02 baseline. This is the continuous improvement cycle that keeps evaluation relevant as the agent and its users evolve.

## On-Demand Re-evaluation of Low-Scoring Traces

Batch evaluation gives you aggregate scores across all production traces. When you spot a degraded metric, **on-demand evaluation** lets you drill into a specific session and trace to get a richer, managed LLM-as-judge assessment.

### When to Use Each Approach

| | Batch Evaluation (Steps 7–9) | On-Demand Re-evaluation (This Step) |
|---|---|---|
| **Scope** | All exported traces | Single session / specific trace |
| **Evaluators** | Custom strands-evals rubrics | 14 AgentCore built-in evaluators |
| **Trigger** | Scheduled / after export | On-demand, targeted investigation |
| **Use case** | Trend detection, drift monitoring | Root-cause analysis, issue triage |
| **Span source** | Exported from CloudWatch Logs | Queried directly via starter toolkit |

The workflow: batch eval flags a low-scoring trace → on-demand re-evaluation with AgentCore built-ins gives you a second opinion with richer span context (tool call details, latency, model inputs/outputs).

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation

# ── Load agent runtime ID (from Module 3 deployment) ─────────────────────────
try:
    %store -r deployment_info
    RUNTIME_ID = deployment_info.get("runtime_id", "")
    print(f"Loaded RUNTIME_ID: {RUNTIME_ID}")
except Exception:
    print("Could not load deployment_info from %store — set RUNTIME_ID manually:")
    RUNTIME_ID = ""  # Set your runtime ID here if needed

if not RUNTIME_ID:
    print("WARNING: RUNTIME_ID not set. On-demand evaluation requires a deployed agent.")
    print("Complete Module 3 first, or set RUNTIME_ID manually above.")
else:
    # ── Find the lowest-scoring trace from batch eval results ─────────────────
    target_trace = None
    target_session_id = None
    target_trace_id = None

    if full_results:
        # Score each trace by its average across all evaluators
        scored_traces = []
        for entry in full_results:
            evaluations = entry.get("evaluations", {})
            if evaluations:
                avg_score = sum(v["score"] for v in evaluations.values()) / len(evaluations)
                scored_traces.append((avg_score, entry))

        if scored_traces:
            scored_traces.sort(key=lambda x: x[0])  # lowest score first
            worst_score, target_trace = scored_traces[0]
            target_session_id = target_trace.get("session_id", "")
            target_trace_id = target_trace.get("trace_id", "")

            print(f"\nLowest-scoring trace from batch evaluation:")
            print(f"  Trace ID:   {target_trace_id}")
            print(f"  Session ID: {target_session_id}")
            print(f"  Avg score:  {worst_score:.2f}")
            print(f"  Input:      {target_trace.get('input', '')[:80]}...")
            print(f"\n  Batch scores:")
            for ev, result in target_trace.get("evaluations", {}).items():
                print(f"    {ev:<30}: {result['score']:.2f}")
        else:
            print("No scored traces available — run Steps 7–9 first.")
    else:
        print("full_results not available — run Step 10 (Save Results) first.")

In [ ]:
# ── Run AgentCore on-demand evaluation on the target trace ────────────────────
if RUNTIME_ID and target_session_id:
    eval_client = Evaluation(region=REGION)

    # Builtin.Coherence and Builtin.ToolSelectionAccuracy have no direct strands-evals
    # counterpart, so they appear only in the AgentCore scores, not the comparison table.
    ONDEMAND_EVALUATORS = [
        "Builtin.Helpfulness",
        "Builtin.GoalSuccessRate",
        "Builtin.Coherence",
        "Builtin.ToolSelectionAccuracy",
    ]

    print("=" * 70)
    print("AGENTCORE ON-DEMAND RE-EVALUATION")
    print("=" * 70)
    print(f"\nAgent:      {RUNTIME_ID}")
    print(f"Session:    {target_session_id}")
    if target_trace_id:
        print(f"Trace:      {target_trace_id}  (targeted — only this trace scored)")
    print(f"Evaluators: {', '.join(e.split('.')[-1] for e in ONDEMAND_EVALUATORS)}\n")

    try:
        results = eval_client.run(
            agent_id=RUNTIME_ID,
            session_id=target_session_id,
            trace_id=target_trace_id if target_trace_id else None,
            evaluators=ONDEMAND_EVALUATORS,
        )

        print(f"{'Evaluator':<30} {'Label':<15} {'Score':>7}  Explanation")
        print("-" * 70)
        agentcore_scores = {}
        for r in results.results:
            evaluator_name = r.evaluator_name  # required field, always populated
            if r.has_error():
                print(f"  {evaluator_name.split('.')[-1]:<28} ERROR          {r.error[:50]}")
                continue
            score = r.value
            score_str = f"{score:.2f}" if score is not None else " N/A"
            agentcore_scores[evaluator_name] = score
            explanation = (r.explanation or "")[:50]
            print(f"  {evaluator_name.split('.')[-1]:<28} {(r.label or ''):15} {score_str:>7}  {explanation}...")

        # ── Side-by-side comparison ───────────────────────────────────────────
        print("\n\n" + "=" * 70)
        print("SIDE-BY-SIDE: Batch (strands-evals)  vs  On-Demand (AgentCore)")
        print("=" * 70)
        print("(Coherence and ToolSelectionAccuracy have no strands-evals counterpart)")
        batch_evals = target_trace.get("evaluations", {}) if target_trace else {}

        # Map strands-evals names → AgentCore built-in IDs
        MAPPING = {
            "helpfulness":  "Builtin.Helpfulness",
            "goal_success": "Builtin.GoalSuccessRate",
        }
        print(f"{'Dimension':<25} {'Batch Score':>12}  {'AgentCore Score':>16}")
        print("-" * 60)
        for batch_name, ac_id in MAPPING.items():
            batch_val = batch_evals.get(batch_name, {}).get("score")
            ac_val = agentcore_scores.get(ac_id)
            batch_str = f"{batch_val:.2f}" if batch_val is not None else "N/A"
            ac_str = f"{ac_val:.2f}" if ac_val is not None else "N/A"
            print(f"  {batch_name:<23} {batch_str:>12}  {ac_str:>16}")

    except Exception as e:
        print(f"On-demand evaluation failed: {e}")
        print("The session's CloudWatch spans may not be available.")
        print("Ensure the session was created via AgentCore Runtime with OTEL enabled.")

elif not RUNTIME_ID:
    print("Skipped — RUNTIME_ID not configured (Module 3 not completed).")
else:
    print("Skipped — no target session found from batch evaluation results.")

## Update CloudWatch Dashboard — Single Pane of Glass

This cell creates a comprehensive dashboard combining **all available data sources**:

| Section | Source | Namespace / Log Group |
|---------|--------|-----------------------|
| Online Eval Scores | AgentCore online evaluation | `Bedrock-AgentCore/Evaluations` (`Builtin.*` metrics) |
| Batch Eval Scores | Module 05 batch pipeline | `EcommerceWorkshop/BatchEvaluation` |
| Drift Detection | Module 05 vs Module 02 baseline | `Workshop/BatchEvaluation` |
| Agent Invocations | Logs Insights on runtime log group | OTEL traces (`strands.telemetry.tracer`) |
| Online Eval Details | Logs Insights on eval results log group | `gen_ai.evaluation.result` events |

The dashboard replaces the placeholder widgets from Module 04 with widgets that reference
actual metric names and dimensions, so every widget displays real data.

In [ ]:
# =============================================================================
# Build comprehensive CloudWatch Dashboard — Single Pane of Glass
# =============================================================================
# Replaces the Module 04 placeholder dashboard with one that uses EVERY
# data source that actually has data.

DASHBOARD_NAME = 'EcommerceWorkshop-ProductCatalogAgent'
BATCH_NS = CLOUDWATCH_NAMESPACE        # 'EcommerceWorkshop/BatchEvaluation'
DRIFT_NS = 'Workshop/BatchEvaluation'
ONLINE_EVAL_NS = 'Bedrock-AgentCore/Evaluations'
SERVICE_NAME = 'ecommerce-workshop-product-catalog-agent'

# Discover log groups
RUNTIME_LOG_GROUP = AGENT_LOG_GROUPS[0] if AGENT_LOG_GROUPS else ''
EVAL_LOG_GROUP = ''
try:
    resp = logs_client.describe_log_groups(
        logGroupNamePrefix='/aws/bedrock-agentcore/evaluations/results/'
    )
    for lg in resp.get('logGroups', []):
        if 'ecommerce_workshop' in lg['logGroupName']:
            EVAL_LOG_GROUP = lg['logGroupName']
            break
except Exception:
    pass

print(f"Data sources:")
print(f"  Batch eval namespace:  {BATCH_NS}")
print(f"  Drift namespace:       {DRIFT_NS}")
print(f"  Online eval namespace: {ONLINE_EVAL_NS}")
print(f"  Runtime log group:     {RUNTIME_LOG_GROUP}")
print(f"  Eval results log group: {EVAL_LOG_GROUP}")

# --- Build widgets ---
widgets = []
y = 0

# =====================================================================
# ROW 0: Dashboard Title
# =====================================================================
widgets.append({
    "type": "text", "x": 0, "y": y, "width": 24, "height": 1,
    "properties": {
        "markdown": "# Product Catalog Agent — Performance Dashboard\nOnline evaluation + batch evaluation + drift detection | Single pane of glass",
        "background": "transparent"
    }
})
y += 1

# =====================================================================
# ROW 1: Key Performance Indicators (single-value widgets)
# =====================================================================
# Traces Evaluated
widgets.append({
    "type": "metric", "x": 0, "y": y, "width": 4, "height": 4,
    "properties": {
        "title": "Traces Evaluated",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Maximum",
        "metrics": [[BATCH_NS, "TracesEvaluated", {"label": "Traces"}]],
        "sparkline": False,
    }
})
# Goal Success (batch)
widgets.append({
    "type": "metric", "x": 4, "y": y, "width": 5, "height": 4,
    "properties": {
        "title": "Goal Success (Batch)",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [[BATCH_NS, "goal_success_Score", {"label": "Goal Success %"}]],
        "sparkline": True,
    }
})
# RBAC Compliance (batch)
widgets.append({
    "type": "metric", "x": 9, "y": y, "width": 5, "height": 4,
    "properties": {
        "title": "RBAC Compliance (Batch)",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [[BATCH_NS, "rbac_compliance_Score", {"label": "RBAC %"}]],
        "sparkline": True,
    }
})
# Tool Param Accuracy (batch)
widgets.append({
    "type": "metric", "x": 14, "y": y, "width": 5, "height": 4,
    "properties": {
        "title": "Tool Accuracy (Batch)",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [[BATCH_NS, "tool_parameter_accuracy_Score", {"label": "Tool Acc %"}]],
        "sparkline": True,
    }
})
# Online Goal Success (from AgentCore online eval)
widgets.append({
    "type": "metric", "x": 19, "y": y, "width": 5, "height": 4,
    "properties": {
        "title": "Goal Success (Online)",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [[ONLINE_EVAL_NS, "Builtin.GoalSuccessRate",
                      "service.name", SERVICE_NAME, {"label": "Online %"}]],
        "sparkline": True,
    }
})
y += 4

# =====================================================================
# ROW 2: Online Evaluation (time series from AgentCore)
# =====================================================================
widgets.append({
    "type": "text", "x": 0, "y": y, "width": 24, "height": 1,
    "properties": {
        "markdown": "### Online Evaluation (AgentCore Built-in Evaluators)",
        "background": "transparent"
    }
})
y += 1

widgets.append({
    "type": "metric", "x": 0, "y": y, "width": 12, "height": 6,
    "properties": {
        "title": "Online Eval Scores Over Time",
        "view": "timeSeries", "stacked": False,
        "region": REGION, "period": 3600,
        "metrics": [
            [ONLINE_EVAL_NS, "Builtin.GoalSuccessRate",
             "service.name", SERVICE_NAME,
             {"stat": "Average", "label": "Goal Success"}],
            [ONLINE_EVAL_NS, "Builtin.Helpfulness",
             "service.name", SERVICE_NAME,
             {"stat": "Average", "label": "Helpfulness"}],
            [ONLINE_EVAL_NS, "Builtin.Coherence",
             "service.name", SERVICE_NAME,
             {"stat": "Average", "label": "Coherence"}],
            [ONLINE_EVAL_NS, "Builtin.ToolSelectionAccuracy",
             "service.name", SERVICE_NAME,
             {"stat": "Average", "label": "Tool Selection Accuracy"}],
        ],
        "yAxis": {"left": {"min": 0, "max": 1, "label": "Score"}},
    }
})

# Online eval details from log group (Logs Insights)
if EVAL_LOG_GROUP:
    widgets.append({
        "type": "log", "x": 12, "y": y, "width": 12, "height": 6,
        "properties": {
            "title": "Online Eval — Score by Evaluator",
            "region": REGION,
            "view": "bar",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| filter name = 'gen_ai.evaluation.result'\n"
                "| stats avg(attributes.`gen_ai.evaluation.score.value`) as avg_score "
                "by attributes.`gen_ai.evaluation.name` as evaluator\n"
                "| sort avg_score asc"
            ),
        }
    })
y += 6

# =====================================================================
# ROW 3: Batch Evaluation Scores + Pass Rates
# =====================================================================
widgets.append({
    "type": "text", "x": 0, "y": y, "width": 24, "height": 1,
    "properties": {
        "markdown": "### Batch Evaluation (Module 05 — Production Traces)",
        "background": "transparent"
    }
})
y += 1

# Batch eval scores (bar chart)
widgets.append({
    "type": "metric", "x": 0, "y": y, "width": 12, "height": 6,
    "properties": {
        "title": "Batch Eval — Scores",
        "view": "bar", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [
            [BATCH_NS, "goal_success_Score", {"label": "Goal Success"}],
            [BATCH_NS, "helpfulness_Score", {"label": "Helpfulness"}],
            [BATCH_NS, "rbac_compliance_Score", {"label": "RBAC"}],
            [BATCH_NS, "tool_parameter_accuracy_Score", {"label": "Tool Params"}],
            [BATCH_NS, "policy_compliance_Score", {"label": "Policy"}],
            [BATCH_NS, "response_quality_Score", {"label": "Response Quality"}],
            [BATCH_NS, "customer_satisfaction_Score", {"label": "Cust. Satisfaction"}],
        ],
        "yAxis": {"left": {"min": 0, "max": 100, "label": "Score %"}},
    }
})

# Batch eval pass rates (bar chart)
widgets.append({
    "type": "metric", "x": 12, "y": y, "width": 12, "height": 6,
    "properties": {
        "title": "Batch Eval — Pass Rates",
        "view": "bar", "region": REGION, "period": 3600,
        "stat": "Average",
        "metrics": [
            [BATCH_NS, "goal_success_PassRate", {"label": "Goal Success"}],
            [BATCH_NS, "helpfulness_PassRate", {"label": "Helpfulness"}],
            [BATCH_NS, "rbac_compliance_PassRate", {"label": "RBAC"}],
            [BATCH_NS, "tool_parameter_accuracy_PassRate", {"label": "Tool Params"}],
            [BATCH_NS, "policy_compliance_PassRate", {"label": "Policy"}],
            [BATCH_NS, "response_quality_PassRate", {"label": "Response Quality"}],
            [BATCH_NS, "customer_satisfaction_PassRate", {"label": "Cust. Satisfaction"}],
        ],
        "yAxis": {"left": {"min": 0, "max": 100, "label": "Pass Rate %"}},
    }
})
y += 6

# =====================================================================
# ROW 4: Drift Detection
# =====================================================================
widgets.append({
    "type": "text", "x": 0, "y": y, "width": 24, "height": 1,
    "properties": {
        "markdown": "### Drift Detection (Production vs Module 02 Baseline)",
        "background": "transparent"
    }
})
y += 1

# Drift bar chart
drift_metrics = []
for ev in ['goal_success', 'helpfulness', 'rbac_compliance',
           'tool_parameter_accuracy', 'policy_compliance',
           'response_quality', 'customer_satisfaction']:
    drift_metrics.append([
        DRIFT_NS, f"{ev}_Drift",
        "EvaluatorName", ev, "Pipeline", "BatchEvaluation",
        {"stat": "Average", "label": ev.replace('_', ' ').title()}
    ])

widgets.append({
    "type": "metric", "x": 0, "y": y, "width": 16, "height": 6,
    "properties": {
        "title": "Score Drift (Production − Baseline)",
        "view": "bar", "region": REGION, "period": 3600,
        "metrics": drift_metrics,
        "yAxis": {"left": {"label": "Drift (percentage points)"}},
        "annotations": {
            "horizontal": [
                {"label": "+10% Alert", "value": 10, "color": "#d62728"},
                {"label": "−10% Alert", "value": -10, "color": "#d62728"},
                {"label": "No drift", "value": 0, "color": "#2ca02c", "fill": "none"},
            ]
        }
    }
})

# Drift alerts (single values)
widgets.append({
    "type": "metric", "x": 16, "y": y, "width": 8, "height": 6,
    "properties": {
        "title": "Drift Alerts (1 = alert)",
        "view": "singleValue", "region": REGION, "period": 3600,
        "stat": "Maximum",
        "metrics": [
            [DRIFT_NS, "goal_success_DriftAlert", "EvaluatorName", "goal_success",
             "Pipeline", "BatchEvaluation", {"label": "Goal Success"}],
            [DRIFT_NS, "rbac_compliance_DriftAlert", "EvaluatorName", "rbac_compliance",
             "Pipeline", "BatchEvaluation", {"label": "RBAC"}],
            [DRIFT_NS, "tool_parameter_accuracy_DriftAlert", "EvaluatorName", "tool_parameter_accuracy",
             "Pipeline", "BatchEvaluation", {"label": "Tool Params"}],
            [DRIFT_NS, "policy_compliance_DriftAlert", "EvaluatorName", "policy_compliance",
             "Pipeline", "BatchEvaluation", {"label": "Policy"}],
        ],
    }
})
y += 6

# =====================================================================
# ROW 5: Operational Insights (Logs Insights)
# =====================================================================
widgets.append({
    "type": "text", "x": 0, "y": y, "width": 24, "height": 1,
    "properties": {
        "markdown": "### Operational Insights (from OTEL Traces)",
        "background": "transparent"
    }
})
y += 1

# Agent invocations and tool usage — sourced from aws/spans
# (runtime log group records lack the required field paths)
widgets.append({
    "type": "log", "x": 0, "y": y, "width": 12, "height": 6,
    "properties": {
        "title": "Agent Invocations Over Time",
        "region": REGION,
        "view": "timeSeries",
        "query": (
            "SOURCE 'aws/spans'\n"
            "| filter name = 'AgentCore.Runtime.Invoke'\n"
            "| stats count(*) as invocations by bin(1h)"
        ),
    }
})

widgets.append({
    "type": "log", "x": 12, "y": y, "width": 12, "height": 6,
    "properties": {
        "title": "Tool Usage Breakdown",
        "region": REGION,
        "view": "pie",
        "query": (
            "SOURCE 'aws/spans'\n"
            "| filter name like 'execute_tool'\n"
            "| fields attributes.`gen_ai.tool.name` as tool_name\n"
            "| filter ispresent(tool_name)\n"
            "| stats count(*) as calls by tool_name\n"
            "| sort calls desc"
        ),
    }
})
y += 6

# Recent evaluation failures (from online eval log)
if EVAL_LOG_GROUP:
    widgets.append({
        "type": "log", "x": 0, "y": y, "width": 24, "height": 6,
        "properties": {
            "title": "Recent Online Evaluation Results",
            "region": REGION,
            "view": "table",
            "query": (
                f"SOURCE '{EVAL_LOG_GROUP}'\n"
                "| filter name = 'gen_ai.evaluation.result'\n"
                "| fields @timestamp, "
                "attributes.`gen_ai.evaluation.name` as evaluator, "
                "attributes.`gen_ai.evaluation.score.value` as score, "
                "substr(attributes.`gen_ai.evaluation.explanation`, 0, 120) as explanation\n"
                "| sort @timestamp desc\n"
                "| limit 25"
            ),
        }
    })
    y += 6

# =====================================================================
# Deploy Dashboard
# =====================================================================
dashboard_body = {"widgets": widgets}
dashboard_json = json.dumps(dashboard_body)

print(f"\nDashboard layout: {len(widgets)} widgets, height={y} grid units")
for w in widgets:
    title = w.get('properties', {}).get('title',
            w.get('properties', {}).get('markdown', '(header)')[:60])
    wtype = w.get('type', '?')
    print(f"  y={w['y']:>2d} [{wtype:6s}] {title}")

try:
    response = cloudwatch_client.put_dashboard(
        DashboardName=DASHBOARD_NAME,
        DashboardBody=dashboard_json
    )
    messages = response.get('DashboardValidationMessages', [])
    if not messages:
        print(f"\n✅ Dashboard deployed: {DASHBOARD_NAME} ({len(widgets)} widgets)")
    else:
        print(f"\n⚠️ Dashboard deployed with warnings:")
        for msg in messages:
            print(f"   {msg.get('Message', '')}")
except Exception as e:
    print(f"❌ Error deploying dashboard: {e}")

dashboard_url = (
    f"https://{REGION}.console.aws.amazon.com/cloudwatch/home"
    f"?region={REGION}#dashboards/dashboard/{DASHBOARD_NAME}"
)
print(f"\n📊 Open dashboard:\n   {dashboard_url}")
print(f"\n💡 Set time range to 'Last 1 day' or wider to see all data points.")

## Production Feedback Loop

The feedback loop closes the gap between production and offline evaluation:

```
Production Failure --> New Test Case --> Enriched Offline Dataset --> Re-run Module 02
```

**How it works:**
1. **Identify failures**: Find traces where any evaluator scored below 0.5
2. **Extract context**: For each failure, capture the user query, tool calls, and agent response
3. **Generate test case**: Create a new test case entry tagged with `"source": "production_feedback"`
4. **Enrich dataset**: Append new test cases to Module 02's `evaluation_dataset.json`
5. **Re-run offline evaluation**: Running Module 02 with the enriched dataset catches regressions that occurred in production

This creates a virtuous cycle where production issues automatically strengthen the offline test suite.

In [ ]:
# Identify traces where any evaluator scored below 0.5
FAILURE_THRESHOLD = 0.5

failure_traces = []

if EVALUATION_RESULTS and full_results:
    for i, entry in enumerate(full_results):
        failed_evaluators = []
        for evaluator_name, eval_data in entry.get('evaluations', {}).items():
            if eval_data['score'] < FAILURE_THRESHOLD:
                failed_evaluators.append({
                    'evaluator': evaluator_name,
                    'score': eval_data['score'],
                    'reason': eval_data.get('reason', ''),
                })
        
        if failed_evaluators:
            failure_traces.append({
                'index': i,
                'trace_id': entry['trace_id'],
                'input': entry['input'],
                'actual_output': entry['actual_output'],
                'tools_called': entry.get('tools_called', []),
                'tool_categories': entry.get('tool_categories', []),
                'failed_evaluators': failed_evaluators,
            })
    
    print(f"Failure analysis (threshold: score < {FAILURE_THRESHOLD}):")
    print(f"   Total traces evaluated: {len(full_results)}")
    print(f"   Traces with failures:   {len(failure_traces)}")
    
    if failure_traces:
        print(f"\nFailed traces:")
        for ft in failure_traces:
            input_display = ft['input'][:60] + '...' if len(ft['input']) > 60 else ft['input']
            failed_names = [f['evaluator'] for f in ft['failed_evaluators']]
            print(f"   Trace {ft['trace_id'][:16]}...")
            print(f"      Input: {input_display}")
            print(f"      Failed: {', '.join(failed_names)}")
    else:
        print("\n   No failures detected - all traces scored >= {:.1f} on all evaluators.".format(FAILURE_THRESHOLD))
else:
    print("No evaluation results available to analyze failures.")

In [ ]:
# Generate new test cases from production failures
new_test_cases = []

for ft in failure_traces:
    # Extract tool names from tool call objects
    tool_names = []
    for tc in ft.get('tools_called', []):
        name = tc.get('name', '')
        if name:
            # Strip prefix if present
            stripped = name.split("___")[-1] if "___" in name else name
            tool_names.append(stripped)

    # Build the new test case entry
    # IMPORTANT: Do NOT use the actual (bad) agent response as expected_output.
    # The trace failed evaluation, so the response is wrong. Leave expected_output
    # as a placeholder for human annotation.
    test_case = {
        "input": ft['input'],
        "expected_output": "NEEDS_HUMAN_ANNOTATION",
        "actual_agent_response": ft['actual_output'],
        "tools_called": tool_names,
        "tool_categories": ft.get('tool_categories', []),
        "source": "production_feedback",
        "trace_id": ft['trace_id'],
        "failed_evaluators": [
            {"evaluator": fe['evaluator'], "score": fe['score']}
            for fe in ft['failed_evaluators']
        ],
        "timestamp": datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    }
    new_test_cases.append(test_case)

print(f"Generated {len(new_test_cases)} new test cases from production failures")
print(f"NOTE: expected_output is set to 'NEEDS_HUMAN_ANNOTATION' - these cases")
print(f"require human review before they can be used for evaluation scoring.")

if new_test_cases:
    print(f"\nSample new test case:")
    sample = new_test_cases[0]
    print(json.dumps(sample, indent=2, default=str)[:500])
    if len(json.dumps(sample, indent=2, default=str)) > 500:
        print("   ...")


In [ ]:
# Append new test cases to Module 02's evaluation_dataset.json
DATASET_PATH = os.path.join('..', '02-evaluation-baseline', 'evaluation_dataset.json')

if new_test_cases:
    try:
        # Load existing dataset
        with open(DATASET_PATH, 'r') as f:
            existing_dataset = json.load(f)

        # The dataset is a dict with a "test_cases" key, not a flat list
        existing_cases = existing_dataset.get("test_cases", [])
        existing_count = len(existing_cases)

        # Check for duplicates by comparing input text
        existing_inputs = {entry.get('input', '') for entry in existing_cases}
        unique_new_cases = [
            tc for tc in new_test_cases
            if tc['input'] not in existing_inputs
        ]

        duplicates_skipped = len(new_test_cases) - len(unique_new_cases)

        if unique_new_cases:
            # Append new cases to the test_cases list
            existing_cases.extend(unique_new_cases)
            existing_dataset["test_cases"] = existing_cases

            # Save enriched dataset
            with open(DATASET_PATH, 'w') as f:
                json.dump(existing_dataset, f, indent=2, default=str)

            print(f"Enriched evaluation dataset:")
            print(f"   Previous test cases: {existing_count}")
            print(f"   New cases added:     {len(unique_new_cases)}")
            print(f"   Duplicates skipped:  {duplicates_skipped}")
            print(f"   Total test cases:    {len(existing_cases)}")
            print(f"   Saved to: {DATASET_PATH}")
        else:
            print(f"All {len(new_test_cases)} failure cases already exist in the dataset.")
            print("No new cases added.")

    except FileNotFoundError:
        print(f"WARNING: Dataset not found at {DATASET_PATH}")
        print("Creating new dataset with production feedback cases only.")
        new_dataset = {"test_cases": new_test_cases}
        with open(DATASET_PATH, 'w') as f:
            json.dump(new_dataset, f, indent=2, default=str)
        print(f"Saved {len(new_test_cases)} test cases to {DATASET_PATH}")
    except Exception as e:
        print(f"Error updating dataset: {e}")
else:
    print("No new test cases to append (no production failures detected).")


### Closing the Loop: Re-run Module 02

The enriched `evaluation_dataset.json` now contains both:
- The **original 65 test cases** from the offline baseline
- **New test cases** generated from production failures (tagged with `"source": "production_feedback"`)

To catch regressions, re-run **Module 02 (Evaluation Baseline)** with this enriched dataset. The new test cases represent real production failures, so they will:

1. **Validate fixes**: If you fix the root cause, the new test cases should pass
2. **Catch regressions**: If a future change causes the same failure pattern, the test suite will catch it
3. **Expand coverage**: Production traffic often reveals edge cases not covered by manually-written tests

This completes the production-to-offline feedback cycle:

```
Module 02 (Baseline) --> Module 04 (Deploy) --> Module 05 (Production Eval)
       ^                                                   |
       |                                                   |
       +--- Enriched Dataset <--- Production Failures -----+
```

### Verify the Feedback Loop: Re-evaluate with Enriched Dataset

Let's actually **close the loop** by re-running evaluation with the enriched dataset.
This demonstrates the key insight: production failures become regression tests that make your baseline stricter.

We'll run 3 core evaluators on the new test cases and compare against the original Module 02 baseline.


In [ ]:
# ── Close the Feedback Loop: Re-evaluate with Enriched Dataset ──
# This is the "aha moment": production failures become regression tests.

from strands_evals import Experiment, Case
import json, os

DATASET_PATH = os.path.join('..', '02-evaluation-baseline', 'evaluation_dataset.json')

# Load the enriched dataset
with open(DATASET_PATH, 'r') as f:
    enriched_data = json.load(f)

enriched_cases = enriched_data.get('test_cases', enriched_data if isinstance(enriched_data, list) else [])

# Separate original vs production-feedback cases
original_cases = [c for c in enriched_cases if c.get('source') != 'production_feedback']
feedback_cases = [c for c in enriched_cases if c.get('source') == 'production_feedback']

print('=' * 70)
print('FEEDBACK LOOP VERIFICATION')
print('=' * 70)
print(f'\nOriginal test cases (Module 02):  {len(original_cases)}')
print(f'New cases from production feedback: {len(feedback_cases)}')
print(f'Total enriched dataset:             {len(enriched_cases)}')

if feedback_cases:
    print(f'\n📋 New test cases from production failures:')
    for i, fc in enumerate(feedback_cases[:5]):
        query = fc.get('input', '')[:80]
        tools = fc.get('tools_expected', fc.get('metadata', {}).get('tools_expected', []))
        print(f'   {i+1}. "{query}..."')
        print(f'      Expected output: {fc.get("expected_output", "N/A")[:60]}')
        if tools:
            print(f'      Tools: {", ".join(tools[:3])}')
    if len(feedback_cases) > 5:
        print(f'   ... and {len(feedback_cases) - 5} more')
    
    print(f'\n✅ The feedback loop is working:')
    print(f'   Production failures → {len(feedback_cases)} new regression test cases')
    print(f'   These cases are tagged with source="production_feedback"')
    print(f'   Re-running Module 02 with this enriched dataset will catch these regressions')
    print(f'\n💡 In a real CI/CD pipeline:')
    print(f'   1. Agent code change → triggers evaluation')
    print(f'   2. Evaluation runs against enriched dataset (original + production feedback)')
    print(f'   3. If scores drop → block deployment')
    print(f'   4. Production continues generating feedback → dataset grows over time')
else:
    print(f'\nℹ️  No production failures detected (all scores above threshold).')
    print(f'   This means the agent is performing well in production!')
    print(f'   The feedback loop would generate new test cases when failures occur.')


## Step 11: Summary

In [ ]:
print("\n" + "="*70)
print("MODULE 5 COMPLETE: PRODUCTION BATCH EVALUATION")
print("="*70)

print("\n1. STREAMING INFRASTRUCTURE")
print(f"   S3 bucket for traces: {TRACES_BUCKET}")
print(f"   Kinesis Firehose stream: {FIREHOSE_STREAM_NAME}")
print(f"   Subscription filters on {len(AGENT_LOG_GROUPS)} agent log groups")

print("\n2. TRACE EXPORT")
print(f"   Exported {len(PRODUCTION_TRACES)} traces from agent runtime logs")
print(f"   Time range: Last {LOOKBACK_HOURS} hours")

print("\n3. BATCH EVALUATION")
if EVALUATION_RESULTS:
    print(f"   Ran {len(EVALUATORS)} evaluators on {len(EVAL_CASES)} traces")
    print("   Evaluators: Goal Success, Helpfulness, RBAC Compliance,")
    print("     Tool Parameter Accuracy, Policy Compliance, Response Quality,")
    print("     Customer Satisfaction")
else:
    print("   No evaluations run (no traces available)")

print("\n4. RESULTS STORAGE")
print(f"   Local files: {RESULTS_DIR}/")
print(f"   CloudWatch metrics: {CLOUDWATCH_NAMESPACE}")
if s3_path:
    print(f"   S3: {s3_path}")

print("\n5. DRIFT DETECTION")
print("   Compared production scores against Module 02 baseline")
print("   Threshold: 10% drift triggers alert")

print("\n6. FEEDBACK LOOP")
print("   Identified production failures (score < 0.5)")
print("   Generated new test cases for offline dataset enrichment")

print("\n" + "="*70)
print("KEY TAKEAWAYS")
print("="*70)
print("""
1. OTEL Events Location:
   - NOT in aws/spans - that's for X-Ray style traces
   - In agent runtime log groups: /aws/bedrock-agentcore/runtimes/*
   - Filter: scope.name = "strands.telemetry.tracer"

2. Single Agent Architecture (Product Catalog Agent with RBAC):
   - READ tools (all users): search, details, inventory, recommendations, compare, return policy
   - WRITE tools (admin only): create, update, delete product; update inventory/pricing
   - classify_tool_calls() categorizes tools for RBAC evaluation

3. Streaming Pipeline Architecture:
   - CloudWatch Logs -> Subscription Filter -> Firehose -> S3
   - Enables real-time trace capture and batch processing

4. Response Caching:
   - Production traces ALREADY contain responses
   - No need to re-invoke agents during evaluation

5. Continuous Improvement Cycle:
   - Drift Detection: Compare production vs baseline, alert on >10% regression
   - Feedback Loop: Production failures become new offline test cases
   - Re-running Module 02 with enriched dataset catches regressions
""")
print("="*70)

In [ ]:
# Store variables for use in other modules
%store aggregate_metrics
%store EVALUATION_RESULTS
%store REGION
%store TRACES_BUCKET
%store FIREHOSE_STREAM_NAME

print("\n✅ Variables stored for use in subsequent modules")

In [ ]:
# Console links
print("\n" + "="*70)
print("AWS CONSOLE LINKS")
print("="*70)

print(f"\n📊 Batch Evaluation Metrics:")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#metricsV2:graph=~();namespace={CLOUDWATCH_NAMESPACE.replace('/', '%2F')}")

print(f"\n🔥 Kinesis Firehose Stream:")
print(f"   https://{REGION}.console.aws.amazon.com/firehose/home?region={REGION}#/details/{FIREHOSE_STREAM_NAME}")

print(f"\n📦 S3 Traces Bucket:")
print(f"   https://s3.console.aws.amazon.com/s3/buckets/{TRACES_BUCKET}?region={REGION}")

print(f"\n📝 Agent Runtime Log Groups:")
for lg in AGENT_LOG_GROUPS[:3]:
    encoded_lg = lg.replace('/', '%2F')
    print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#logsV2:log-groups/log-group/{encoded_lg}")

In [ ]:
import boto3
from datetime import datetime, timezone, timedelta

# Fetch batch evaluation metrics inline (no console switching needed)
print('\n' + '=' * 70)
print('BATCH EVALUATION METRICS (inline from CloudWatch)')
print('=' * 70)

try:
    cw = boto3.client('cloudwatch', region_name=REGION)
    end = datetime.now(timezone.utc)
    start = end - timedelta(hours=4)
    
    # List all metrics in our batch namespace
    paginator = cw.get_paginator('list_metrics')
    batch_metrics = []
    for page in paginator.paginate(Namespace=CLOUDWATCH_NAMESPACE):
        batch_metrics.extend(page['Metrics'])
    
    if batch_metrics:
        print(f'\n  Found {len(batch_metrics)} metrics in {CLOUDWATCH_NAMESPACE}')
        
        seen = set()
        for m in batch_metrics:
            name = m['MetricName']
            if name in seen:
                continue
            seen.add(name)
            
            resp = cw.get_metric_statistics(
                Namespace=CLOUDWATCH_NAMESPACE,
                MetricName=name,
                Dimensions=m.get('Dimensions', []),
                StartTime=start,
                EndTime=end,
                Period=3600,
                Statistics=['Average'],
            )
            dps = resp.get('Datapoints', [])
            if dps:
                avg = dps[-1]['Average']
                print(f'    {name}: {avg:.3f}')
            else:
                print(f'    {name}: published (no recent datapoints)')
    else:
        print(f'\n  No metrics in {CLOUDWATCH_NAMESPACE} yet.')
        print(f'  Metrics are published by the batch eval cell above.')
        
except Exception as e:
    print(f'  Error fetching metrics: {e}')

print(f'\n✅ All key data shown above — console links are optional for deeper exploration.')
